In [2]:
!pip install geopandas
import geopandas as gpd

# GADM level 2 = districts
gadm_url = "https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_IND_2.json"
india_districts = gpd.read_file(gadm_url)
india_districts.to_file("india_districts.gpkg", driver="GPKG")  # cache locally

In [3]:
"""
GFM Datewise Combined TIFs
- Downloads ensemble flood extent + reference water tiles
- Merges tiles per date
- Combines flood + ref water into a classified TIF per date
- Reprojects to EPSG:4326 per date
"""

# ============================================================
# INSTALLS
# ============================================================

!pip install pystac_client

# ============================================================
# IMPORTS
# ============================================================

import os
import re
import numpy as np
from urllib.parse import urlparse
from collections import defaultdict

import requests
import rasterio
from rasterio.merge import merge
from rasterio.mask import mask
from rasterio.warp import (
    calculate_default_transform,
    reproject,
    Resampling
)
from rasterio.crs import CRS
from pystac_client import Client
from shapely.geometry import box, shape, mapping
from pyproj import Transformer
from rasterio.warp import reproject, Resampling

# ============================================================
# CONFIG
# ============================================================

EMAIL    = "sparshsheth1502@gmail.com"
PASSWORD = "a1b2c3d4"

BASE_URL = "https://api.gfm.eodc.eu/v2"
STAC_URL = "https://stac.eodc.eu/api/v1"

DISTRICT_NAME = "Barpeta"    # <-- change this
START_DATE = "2023-01-01"
END_DATE   = "2023-12-31"

# ============================================================
# BBOX FROM DISTRICT
# ============================================================

def get_bbox_from_district(district_name, padding=0.1):
    r = requests.get(
        "https://nominatim.openstreetmap.org/search",
        params={"q": district_name, "format": "json", "limit": 1},
        headers={"User-Agent": "gfm-flood-pipeline/1.0"}
    )
    r.raise_for_status()
    results = r.json()
    if not results:
        raise ValueError(f"Not found: {district_name}")
    south, north, west, east = map(float, results[0]["boundingbox"])
    aoi_bbox = [west - padding, south - padding, east + padding, north + padding]
    print(f"bbox for '{district_name}': {aoi_bbox}")
    return aoi_bbox

def get_district_info(district_name, padding=0.1):
    """
    district_name: just the district name e.g. 'Ernakulam'
    """
    gdf = gpd.read_file("india_districts.gpkg")

    # Match district name (case-insensitive)
    match = gdf[gdf["NAME_2"].str.lower() == district_name.split(",")[0].strip().lower()]

    if match.empty:
        # Print available names to help debug
        print("No match found. Similar names:")
        print(gdf[gdf["NAME_2"].str.lower().str.contains(
            district_name.split(",")[0].strip().lower()
        )]["NAME_2"].values)
        raise ValueError(f"District '{district_name}' not found in GADM")

    row = match.iloc[0]
    geom = mapping(row.geometry)   # GeoJSON-like dict for rasterio
    bounds = row.geometry.bounds   # (minx, miny, maxx, maxy)

    aoi_bbox = [
        bounds[0] - padding,
        bounds[1] - padding,
        bounds[2] + padding,
        bounds[3] + padding,
    ]

    print(f"Matched district : {row['NAME_2']}, {row['NAME_1']}")
    print(f"Geometry type    : {row.geometry.geom_type}")
    print(f"bbox (padded)    : {aoi_bbox}")

    return aoi_bbox, geom

from rasterio.mask import mask as rio_mask

def clip_to_district(input_tif, output_tif, district_geom):
    from rasterio.mask import mask as rio_mask
    from shapely.geometry import shape, mapping
    import geopandas as gpd

    geom = shape(district_geom)
    if not geom.is_valid:
        geom = geom.buffer(0)

    gdf = gpd.GeoDataFrame(geometry=[geom], crs="EPSG:4326")

    with rasterio.open(input_tif) as src:
        if src.crs.to_epsg() != 4326:
            gdf = gdf.to_crs(src.crs)

        clipped, transform = rio_mask(
            src,
            [mapping(gdf.geometry.iloc[0])],
            crop=True,
            nodata=255,        # <-- use 255 as nodata, not 0 (0 is a valid class)
            all_touched=True
        )
        meta = src.meta.copy()
        meta.update({
            "height":    clipped.shape[1],
            "width":     clipped.shape[2],
            "transform": transform,
            "nodata":    255,  # <-- declare it so viewers respect it
            "compress":  "lzw"
        })
        with rasterio.open(output_tif, "w", **meta) as dst:
            dst.write(clipped)
            for band in range(clipped.shape[0]):
                dst.set_band_description(band + 1, f"Biweek_{band + 1}")

    print(f"  Saved: {output_tif}")

AOI_BBOX, DISTRICT_GEOM = get_district_info(DISTRICT_NAME)
START_DATE = "2023-01-01"
END_DATE   = "2023-12-31"

DOWNLOAD_DIR = "gfm_downloads"
MERGED_DIR   = "gfm_merged"
OUTPUT_DIR   = "gfm_final"

DOWNLOAD_ASSETS = [
    "ensemble_flood_extent",
    "reference_water_mask",
]

os.makedirs(DOWNLOAD_DIR, exist_ok=True)
os.makedirs(MERGED_DIR,   exist_ok=True)
os.makedirs(OUTPUT_DIR,   exist_ok=True)


# ============================================================
# AUTH
# ============================================================

def get_token():
    print("Authenticating...")
    r = requests.post(
        f"{BASE_URL}/auth/login",
        json={"email": EMAIL, "password": PASSWORD}
    )
    r.raise_for_status()
    print("Authenticated")
    return r.json()["access_token"]


# ============================================================
# SEARCH ENSEMBLE FLOOD ITEMS
# ============================================================

def search_products(token):
    print("\nSearching STAC for ENSEMBLE_FLOOD items...")

    catalog = Client.open(
        STAC_URL,
        request_modifier=lambda req: req.headers.update({
            "Authorization": f"Bearer {token}"
        }) or req
    )

    search = catalog.search(
        collections=["GFM"],
        bbox=AOI_BBOX,
        datetime=f"{START_DATE}/{END_DATE}",
        max_items=10000,
    )

    items = list(search.items())
    print(f"Raw items found: {len(items)}")

    aoi = box(*AOI_BBOX)
    filtered = []

    for item in items:
        try:
            geom = shape(item.geometry)
            if not geom.intersects(aoi):
                continue
            if "ENSEMBLE_FLOOD" not in item.id:
                continue

            filtered.append(item)
        except Exception as e:
            print(f"  Error processing {item.id}: {e}")

    print(f"Filtered items: {len(filtered)}")
    for item in filtered:
        print(f"  {item.id}")

    return filtered

from shapely.ops import unary_union

def check_spatial_coverage(items):
    aoi = box(*AOI_BBOX)
    footprints = []
    for item in items:
        footprints.append(shape(item.geometry))

    union = unary_union(footprints)
    covered = union.intersection(aoi)
    pct = covered.area / aoi.area * 100

    print(f"\nAOI area:       {aoi.area:.4f}")
    print(f"Covered area:   {covered.area:.4f}")
    print(f"Coverage:       {pct:.1f}%")

    uncovered = aoi.difference(union)
    print(f"Uncovered area: {uncovered.area:.4f}")

    return union, uncovered

# ============================================================
# DOWNLOAD FLOOD + REF WATER TILES
# ============================================================

def download_products(items, token):
    """Download all assets. Returns {asset_key: {date: [filepath, ...]}}"""

    headers = {"Authorization": f"Bearer {token}"}
    total   = len(items) * len(DOWNLOAD_ASSETS)
    count   = 0
    downloaded = defaultdict(lambda: defaultdict(list))

    for item in items:
        print(f"\n--- {item.id} ---")

        # Extract YYYYMMDD from item ID e.g. ENSEMBLE_FLOOD_20230608T004142_...
        match = re.search(r'(\d{8})T', item.id)
        if not match:
            print(f"  Skipping {item.id} — no date found in item ID")
            continue
        date = match.group(1)

        for asset_key in DOWNLOAD_ASSETS:
            count += 1

            if asset_key not in item.assets:
                print(f"  [{count}/{total}] MISSING asset '{asset_key}'")
                continue

            url   = item.assets[asset_key].href
            fname = os.path.basename(urlparse(url).path)
            out_path = os.path.join(DOWNLOAD_DIR, fname)

            if os.path.exists(out_path):
                print(f"  [{count}/{total}] Exists: {fname}")
            else:
                print(f"  [{count}/{total}] Downloading [{asset_key}]: {fname}")
                r = requests.get(url, headers=headers, stream=True)
                r.raise_for_status()
                with open(out_path, "wb") as f:
                    for chunk in r.iter_content(1024 * 1024):
                        if chunk:
                            f.write(chunk)
                print(f"  Saved: {out_path}")

            downloaded[asset_key][date].append(out_path)

    return downloaded


# ============================================================
# SEARCH ALL REFERENCE WATER TILES IN AOI
# (catches tiles not covered by ENSEMBLE_FLOOD items)
# ============================================================

def search_reference_water(token):
    print("\nSearching all reference water tiles in AOI...")

    catalog = Client.open(
        STAC_URL,
        request_modifier=lambda req: req.headers.update({
            "Authorization": f"Bearer {token}"
        }) or req
    )

    search = catalog.search(
        collections=["GFM"],
        bbox=AOI_BBOX,
        datetime=f"{START_DATE}/{END_DATE}",
        max_items=10000,
    )

    items    = list(search.items())
    aoi      = box(*AOI_BBOX)
    seen     = set()
    ref_paths = []

    for item in items:
        try:
            geom = shape(item.geometry)
            if not geom.intersects(aoi):
                continue
            if "reference_water_mask" not in item.assets:
                continue

            url   = item.assets["reference_water_mask"].href
            fname = os.path.basename(urlparse(url).path)

            if fname in seen:
                continue
            seen.add(fname)

            out_path = os.path.join(DOWNLOAD_DIR, fname)

            if os.path.exists(out_path):
                print(f"  Exists: {fname}")
            else:
                print(f"  Downloading [reference_water_mask]: {fname}")
                headers = {"Authorization": f"Bearer {token}"}
                r = requests.get(url, headers=headers, stream=True)
                r.raise_for_status()
                with open(out_path, "wb") as f:
                    for chunk in r.iter_content(1024 * 1024):
                        if chunk:
                            f.write(chunk)
                print(f"  Saved: {out_path}")

            ref_paths.append(out_path)

        except Exception as e:
            print(f"  Error: {e}")

    print(f"  Total unique reference water tiles: {len(ref_paths)}")
    return ref_paths


# ============================================================
# MERGE HELPER
# ============================================================

def merge_tifs(file_list, out_path):
    """Merge a list of TIF files into one mosaic."""
    datasets = [rasterio.open(f) for f in file_list]
    mosaic, transform = merge(datasets)

    meta = datasets[0].meta.copy()
    meta.update({
        "driver":    "GTiff",
        "height":    mosaic.shape[1],
        "width":     mosaic.shape[2],
        "transform": transform,
        "compress":  "lzw",
    })

    with rasterio.open(out_path, "w", **meta) as dest:
        dest.write(mosaic)

    for ds in datasets:
        ds.close()

    print(f"    Merged -> {out_path}")
    return out_path


# ============================================================
# MERGE PER DATE
# Returns {date: {"flood": path, "ref_water": path}}
# ============================================================

def merge_per_date(downloaded, all_ref_water_paths):
    """
    - Flood:     merge tiles per date  -> ENSEMBLE_FLOOD_{date}_merged.tif
    - Ref water: merge all tiles once  -> then reuse the same file for every date
    """
    print("\n\nMERGING...\n")

    merged = defaultdict(dict)

    # ---- Reference water: one merged file for all dates ----
    unique_ref = list(dict.fromkeys(all_ref_water_paths))
    ref_merged_path = os.path.join(MERGED_DIR, "REFERENCE_WATER_merged.tif")

    if os.path.exists(ref_merged_path):
        print(f"  Exists: {os.path.basename(ref_merged_path)}")
    else:
        print(f"  Merging {len(unique_ref)} reference water tiles...")
        merge_tifs(unique_ref, ref_merged_path)

    # ---- Flood: per date ----
    flood_by_date = downloaded.get("ensemble_flood_extent", {})

    for date, paths in sorted(flood_by_date.items()):
        unique_paths = list(dict.fromkeys(paths))
        flood_merged_path = os.path.join(MERGED_DIR, f"ENSEMBLE_FLOOD_{date}_merged.tif")

        if os.path.exists(flood_merged_path):
            print(f"  Exists: {os.path.basename(flood_merged_path)}")
        else:
            print(f"  Merging {len(unique_paths)} flood tiles for {date}...")
            merge_tifs(unique_paths, flood_merged_path)

        merged[date]["flood"]     = flood_merged_path
        merged[date]["ref_water"] = ref_merged_path   # same ref water for every date

    return merged


# ============================================================
# STEP 1  COMBINE FLOOD + REF WATER -> CLASSIFIED TIF
# ============================================================
#
# Output classes:
#   0 = nodata
#   1 = flood extent
#   2 = land
#   3 = seasonal water
#   4 = permanent water
#
# ============================================================

def combine_tifs(flood_tif, ref_water_tif, output_tif):
    print(f"\n  Combining -> {os.path.basename(output_tif)}")

    with rasterio.open(flood_tif) as ref:
        ref_meta      = ref.meta.copy()
        ref_height    = ref.height
        ref_width     = ref.width
        ref_crs       = ref.crs
        ref_transform = ref.transform

    combined = np.zeros((ref_height, ref_width), dtype=np.uint16)

    # ---- Flood layer ----
    with rasterio.open(flood_tif) as ds:
        arr = np.zeros((ref_height, ref_width), dtype=np.uint8)
        reproject(
            source=rasterio.band(ds, 1),
            destination=arr,
            src_transform=ds.transform,
            src_crs=ds.crs,
            dst_transform=ref_transform,
            dst_crs=ref_crs,
            resampling=Resampling.nearest
        )
        combined[arr == 1] = 1      # flood pixels -> class 1
        print(f"    Flood unique vals: {np.unique(arr[arr != 255])}")

    # ---- Reference water layer ----
    with rasterio.open(ref_water_tif) as ds:
        arr = np.zeros((ref_height, ref_width), dtype=np.uint8)
        reproject(
            source=rasterio.band(ds, 1),
            destination=arr,
            src_transform=ds.transform,
            src_crs=ds.crs,
            dst_transform=ref_transform,
            dst_crs=ref_crs,
            resampling=Resampling.nearest
        )
        print(f"    Ref water unique vals: {np.unique(arr[arr != 255])}")

        # Only fill pixels NOT already marked as flood
        combined[(combined == 0) & (arr == 0)] = 2   # land
        combined[(combined == 0) & (arr == 2)] = 3   # seasonal water
        combined[(combined == 0) & (arr == 1)] = 4   # permanent water

    ref_meta.update({"count": 1, "dtype": "uint16", "compress": "lzw"})

    with rasterio.open(output_tif, "w", **ref_meta) as dst:
        dst.write(combined, 1)

    print(f"    Saved: {output_tif}")
    return output_tif


# ============================================================
# STEP 2  REPROJECT TO EPSG:4326
# ============================================================

def reproject_to_4326(input_tif, output_tif):
    with rasterio.open(input_tif) as src:
        dst_crs = CRS.from_epsg(4326)
        transform, width, height = calculate_default_transform(
            src.crs, dst_crs, src.width, src.height, *src.bounds
        )
        meta = src.meta.copy()
        meta.update({
            "crs":       dst_crs,
            "transform": transform,
            "width":     width,
            "height":    height,
            "compress":  "lzw"
        })
        with rasterio.open(output_tif, "w", **meta) as dst:
            reproject(
                source=rasterio.band(src, 1),
                destination=rasterio.band(dst, 1),
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=dst_crs,
                resampling=Resampling.nearest
            )
            dst.write_colormap(1, {
                1: (148, 0,   211, 255),  # flood       - violet
                2: (255, 255, 255, 255),  # land        - white
                3: (0,   255, 255, 255),  # seasonal    - cyan
                4: (0,   0,   255, 255),  # permanent   - blue
            })

    print(f"    Reprojected: {output_tif}")
    return output_tif

from datetime import datetime

def create_biweekly_stack(input_dir, output_tif, year):

    biweek_files = {i: [] for i in range(26)}

    year_start = datetime(year, 1, 1)

    # ---------------------------------------
    # Assign date TIFFs to biweeks
    # ---------------------------------------

    for fname in os.listdir(input_dir):

        if not fname.startswith("combined_4326_"):
            continue

        date_str = (
            fname.replace("combined_4326_", "")
                 .replace(".tif", "")
        )

        try:
            d = datetime.strptime(date_str, "%Y%m%d")
        except:
            continue

        if d.year != year:
            continue

        biweek_idx = (d - year_start).days // 14

        if biweek_idx > 25:
            biweek_idx = 25

        biweek_files[biweek_idx].append(
            os.path.join(input_dir, fname)
        )

    # ---------------------------------------
    # Metadata from first file
    # ---------------------------------------

    all_files = [f for files in biweek_files.values() for f in files]

    if not all_files:
        raise ValueError("No classified TIFFs found")

    min_left = min_bottom = float('inf')
    max_right = max_top = float('-inf')
    sample_meta = None

    for f in all_files:
        with rasterio.open(f) as src:
            b = src.bounds
            min_left   = min(min_left,   b.left)
            min_bottom = min(min_bottom, b.bottom)
            max_right  = max(max_right,  b.right)
            max_top    = max(max_top,    b.top)
            if sample_meta is None:
                sample_meta = src.meta.copy()
                res_x = src.res[0]
                res_y = src.res[1]

    from rasterio.transform import from_bounds
    cols = int(round((max_right  - min_left)   / res_x))
    rows = int(round((max_top    - min_bottom) / res_y))
    transform = from_bounds(min_left, min_bottom, max_right, max_top, cols, rows)

    meta = sample_meta.copy()
    meta.update({
        "width":     cols,
        "height":    rows,
        "transform": transform,
    })

    stack = np.zeros(
        (26, rows, cols),
        dtype=np.uint8
    )

    # ---------------------------------------
    # Priority compositing
    #
    # Flood(1)
    #   >
    # Seasonal(3)
    #   >
    # Permanent(4)
    #   >
    # Land(2)
    # ---------------------------------------

    for bw in range(26):

        files = biweek_files[bw]

        print(
            f"Biweek {bw+1}: {len(files)} files"
        )

        if len(files) == 0:
            continue

        max_priority = np.zeros(
            (rows, cols),
            dtype=np.uint8
        )

        for f in files:

            with rasterio.open(f) as src:

                arr = np.zeros(
                    (rows, cols),
                    dtype=np.uint8
                )

                reproject(
                    source=rasterio.band(src, 1),
                    destination=arr,
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=meta["transform"],
                    dst_crs=meta["crs"],
                    resampling=Resampling.nearest
                )

                pri = np.zeros_like(
                    arr,
                    dtype=np.uint8
                )

                pri[arr == 2] = 1   # land
                pri[arr == 4] = 2   # permanent
                pri[arr == 3] = 3   # seasonal
                pri[arr == 1] = 4   # flood

                max_priority = np.maximum(
                    max_priority,
                    pri
                )

        out = np.zeros(
            (rows, cols),
            dtype=np.uint8
        )

        out[max_priority == 1] = 2
        out[max_priority == 2] = 4
        out[max_priority == 3] = 3
        out[max_priority == 4] = 1

        stack[bw] = out

    meta.update(
        count=26,
        dtype="uint8",
        compress="lzw"
    )

    with rasterio.open(
        output_tif,
        "w",
        **meta
    ) as dst:

        for band in range(26):

            dst.write(
                stack[band],
                band + 1
            )

            dst.set_band_description(
                band + 1,
                f"Biweek_{band+1}"
            )

    print(f"\nSaved: {output_tif}")

# ============================================================
# MAIN
# ============================================================

def run():
    token = get_token()

    # -- Download --
    items      = search_products(token)
    if not items:
        print("\nNo products found")
        return

    downloaded        = download_products(items, token)
    all_ref_water_paths = search_reference_water(token)

    items = search_products(token)
    union, uncovered = check_spatial_coverage(items)

    # -- Merge per date --
    merged_by_date = merge_per_date(downloaded, all_ref_water_paths)

    # -- Combine + reproject per date --
    print("\n\nPROCESSING PER DATE...\n")

    for date in sorted(merged_by_date.keys()):
        flood_tif     = merged_by_date[date]["flood"]
        ref_water_tif = merged_by_date[date]["ref_water"]

        print(f"\n========== {date} ==========")

        combined_native = os.path.join(OUTPUT_DIR, f"combined_native_{date}.tif")
        combined_4326   = os.path.join(OUTPUT_DIR, f"combined_4326_{date}.tif")

        if os.path.exists(combined_4326):
            print(f"  Already exists: {os.path.basename(combined_4326)}")
            continue

        # Step 1: combine
        result = combine_tifs(flood_tif, ref_water_tif, combined_native)

        # Step 2: reproject
        if result:
          reproject_to_4326(combined_native, combined_4326)
          os.remove(combined_native)

    stack_tif   = os.path.join(OUTPUT_DIR, f"gfm_biweekly_{START_DATE[:4]}.tif")
    clipped_tif = os.path.join(OUTPUT_DIR, f"gfm_biweekly_{START_DATE[:4]}_clipped.tif")

    create_biweekly_stack(
        input_dir=OUTPUT_DIR,
        output_tif=stack_tif,
        year=int(START_DATE[:4])
    )

    clip_to_district(stack_tif, clipped_tif, DISTRICT_GEOM)

    print("\n\nDONE")
    print(f"Output files in: {OUTPUT_DIR}/")
    for f in sorted(os.listdir(OUTPUT_DIR)):
        print(f"  {f}")


run()

Matched district : Barpeta, Assam
Geometry type    : MultiPolygon
bbox (padded)    : [90.55170000000001, 25.9895, 91.39349999999999, 26.764300000000002]
Authenticating...
Authenticated

Searching STAC for ENSEMBLE_FLOOD items...
Raw items found: 457
Filtered items: 457
  ENSEMBLE_FLOOD_20231227T114935_VV_AS020M_E039N024T3
  ENSEMBLE_FLOOD_20231227T114910_VV_AS020M_E039N024T3
  ENSEMBLE_FLOOD_20231225T120559_VV_AS020M_E039N024T3
  ENSEMBLE_FLOOD_20231225T120534_VV_AS020M_E039N024T3
  ENSEMBLE_FLOOD_20231225T120509_VV_AS020M_E039N024T3
  ENSEMBLE_FLOOD_20231222T234751_VV_AS020M_E039N024T3
  ENSEMBLE_FLOOD_20231222T234726_VV_AS020M_E039N024T3
  ENSEMBLE_FLOOD_20231222T234701_VV_AS020M_E039N024T3
  ENSEMBLE_FLOOD_20231221T000350_VV_AS020M_E039N024T3
  ENSEMBLE_FLOOD_20231221T000325_VV_AS020M_E039N024T3
  ENSEMBLE_FLOOD_20231220T115743_VV_AS020M_E039N024T3
  ENSEMBLE_FLOOD_20231220T115718_VV_AS020M_E039N024T3
  ENSEMBLE_FLOOD_20231220T115653_VV_AS020M_E039N024T3
  ENSEMBLE_FLOOD_20231215T11